# CI/CD Observability Stream

Solo italiano.

Questo notebook è una piccola traccia per la presentazione del progetto TAP.
Segue un evento della pipeline CI/CD mentre attraversa lo stack locale: 
- Jenkins --> OpenTelemetry --> Logstash --> Kafka --> Spark Structured Streaming --> Spark MLlib --> Elasticsearch --> Kibana.



## Setup

Le funzioni sotto sono helper e servono a chiamare Docker Compose, leggere le API HTTP locali e stampare pochi campi utili senza riempire la presentazione di output lunghi.

In [ ]:
from pathlib import Path
import base64, json, subprocess, urllib.request

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
TOPICS = ["cicd.otel.raw", "cicd.otel.processed", "cicd.otel.scored"]


def docker_compose(*args, timeout=20):
    result = subprocess.run(["docker", "compose", *args], cwd=ROOT, text=True,
                            capture_output=True, timeout=timeout)
    return result.stdout.strip()


def get_json(url, user=None, password=None):
    request = urllib.request.Request(url)
    if user:
        token = base64.b64encode(f"{user}:{password}".encode()).decode()
        request.add_header("Authorization", f"Basic {token}")
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.loads(response.read().decode())


def topic_offset(topic):
    output = docker_compose("exec", "-T", "kafka", "/opt/kafka/bin/kafka-get-offsets.sh",
                         "--bootstrap-server", "localhost:9092", "--topic", topic)
    return sum(int(row.rsplit(":", 1)[1]) for row in output.splitlines() if row.startswith(topic))


def read_topic(topic, limit=3):
    output = docker_compose("exec", "-T", "kafka", "/opt/kafka/bin/kafka-console-consumer.sh",
                         "--bootstrap-server", "localhost:9092", "--topic", topic,
                         "--partition", "0", "--offset", str(max(0, topic_offset(topic) - limit)),
                         "--max-messages", str(limit), "--timeout-ms", "5000", timeout=20)
    return [json.loads(row) for row in output.splitlines() if row.startswith("{")]


def show_records(records, fields):
    for index, record in enumerate(records, 1):
        print(f"\nRecord {index}")
        for field in fields:
            if field in record:
                print(f"  {field}: {record[field]}")

print("Project root:", ROOT)

## Stack in esecuzione

Prima di guardare i dati, verifico che tutti i servizi principali siano vivi con `docker compose ps`.

In [3]:
services = {json.loads(row)["Service"]: json.loads(row) for row in docker_compose("ps", "--format", "json").splitlines()}
service_order = ["jenkins", "otel-collector", "logstash", "kafka", "spark-processor",
                 "spark-mllib", "elasticsearch-indexer", "elasticsearch", "kibana"]

for name in service_order:
    service = services.get(name, {})
    print(f"{name:28} {service.get('State', 'missing'):10} {service.get('Status', '')}")

jenkins                      running    Up 48 minutes
otel-collector               running    Up 48 minutes
logstash                     running    Up 48 minutes
kafka                        running    Up 49 minutes (healthy)
spark-processor              running    Up 48 minutes
spark-mllib                  running    Up 48 minutes
elasticsearch-indexer        running    Up 48 minutes
elasticsearch                running    Up 49 minutes (healthy)
kibana                       running    Up 48 minutes


## Jenkins, OpenTelemetry e Logstash

Jenkins simula la pipeline CI/CD. OpenTelemetry scrive segnali in file JSONL, poi Logstash li legge insieme ai log Jenkins e li pubblica nel topic Kafka raw. (`cicd.otel.raw`).

In [4]:
job = get_json("http://localhost:8080/job/demo-ci-observability/api/json?tree=displayName,builds[number,result,duration]{0,3}", "admin", "admin")
print("Jenkins job:", job["displayName"])
for build in job["builds"]:
    print(f"  build #{build['number']}: {build['result']} ({build['duration']} ms)")

otel_files = sorted((ROOT / "otel-output").glob("*.jsonl"))
print("\nOpenTelemetry files:", ", ".join(path.name for path in otel_files))
if otel_files:
    sample = json.loads(next(row for row in otel_files[0].read_text(encoding="utf-8", errors="ignore").splitlines() if row.strip()))
    print("Sample OTLP keys:", ", ".join(sample.keys()))

stats = get_json("http://localhost:9600/_node/stats/pipelines")
events = stats["pipelines"]["main"]["events"]
print("\nLogstash events:", {key: events[key] for key in ["in", "filtered", "out"]})

show_records(read_topic("cicd.otel.raw", 1), ["@timestamp", "event.dataset", "ingestion.component", "otel.signal", "path"])

Jenkins job: demo-ci-observability
  build #683: FAILURE (10802 ms)
  build #682: SUCCESS (13810 ms)
  build #681: SUCCESS (17921 ms)

OpenTelemetry files: logs.jsonl, metrics.jsonl, traces.jsonl
Sample OTLP keys: resourceLogs

Logstash events: {'in': 507, 'filtered': 507, 'out': 141}

Record 1
  @timestamp: 2026-05-22T11:18:20.566388252Z
  event.dataset: jenkins.otel.raw
  ingestion.component: logstash
  otel.signal: metrics
  path: /otel-output/metrics.jsonl


## Kafka

Kafka fa da passaggio stabile tra gli stadi. Qui si vede il percorso principale dei topic: raw, processed e scored.

In [5]:
print("CI/CD topics:")
for topic in docker_compose("exec", "-T", "kafka", "/opt/kafka/bin/kafka-topics.sh",
                         "--bootstrap-server", "localhost:9092", "--list").splitlines():
    if topic.startswith("cicd."):
        print(" ", topic)

print("\nFinal offsets:")
for topic in TOPICS:
    print(f"  {topic:22} {topic_offset(topic)}")

CI/CD topics:
  cicd.otel.processed
  cicd.otel.raw
  cicd.otel.scored

Final offsets:
  cicd.otel.raw          14867
  cicd.otel.processed    3236
  cicd.otel.scored       3236


## Spark Structured Streaming

Il primo job Spark legge `cicd.otel.raw`, estrae i campi CI/CD interessanti e scrive eventi pi? puliti in `cicd.otel.processed`.

In [7]:
processed = read_topic("cicd.otel.processed", 5)
show_records(processed, ["processing_component", "job_name", "build_number", "ci_stage",
                         "ci_status", "event_kind", "signal_domain", "signal_name",
                         "signal_value", "severity_level", "feature_overall_pressure"])

print("\nEvent kinds:", sorted({record.get("event_kind") for record in processed if record.get("event_kind")}))


Record 1
  processing_component: spark-structured-streaming
  job_name: demo-ci-observability
  build_number: 682
  ci_stage: checkout
  ci_status: success
  event_kind: stage_result
  signal_domain: source_control
  signal_name: scm_latency
  signal_value: 3600.0
  severity_level: normal
  feature_overall_pressure: 0.4074074074074074

Record 2
  processing_component: spark-structured-streaming
  job_name: demo-ci-observability
  build_number: 682
  ci_stage: test
  ci_status: passed
  event_kind: stage_result
  signal_domain: test_quality
  signal_name: test_duration_ms
  signal_value: 6060.0
  severity_level: normal
  feature_overall_pressure: 0.505

Record 3
  processing_component: spark-structured-streaming
  job_name: demo-ci-observability
  build_number: 682
  ci_stage: build
  ci_status: success
  event_kind: stage_result
  signal_domain: build
  signal_name: compile_time_ms
  signal_value: 6900.0
  severity_level: normal
  feature_overall_pressure: 0.575

Record 4
  processing

## Spark MLlib

Il secondo job Spark legge gli eventi processed, applica un piccolo modello di Logistic Regression e pubblica il risultato in `cicd.otel.scored`.

Nota bene: `is_failure` indica un errore già osservato da Jenkins (cioè una build fallita, con pipeline arrestata allo step del fallimento).

Invece, `ml_stage_failure_warning` indica un avviso predittivo emesso dal modello.

In [8]:
scored = read_topic("cicd.otel.scored", 8)
show_records(scored, ["processing_component", "ml_model_name", "ml_model_version",
                      "ml_stage_failure_warning", "warning_level", "warning_type",
                      "warning_title", "recommended_action", "build_number", "ci_stage",
                      "signal_name", "is_failure"])

print("\nPredictive warnings:", sum(bool(record.get("ml_stage_failure_warning")) for record in scored))
print("Observed failures:", sum(bool(record.get("is_failure")) for record in scored))


Record 1
  processing_component: spark-mllib-stage-failure-prediction
  ml_model_name: tap-ci-stage-failure-logistic-regression
  ml_model_version: cicd-stage-failure-logreg-v1
  warning_level: warning
  warning_type: predicted_stage_failure
  warning_title: Stage may fail demo-ci-observability #682 package
  recommended_action: verify_artifact_size_and_checksum
  build_number: 682
  ci_stage: package
  signal_name: artifact_size_mb
  is_failure: False

Record 2
  processing_component: spark-mllib-stage-failure-prediction
  ml_model_name: tap-ci-stage-failure-logistic-regression
  ml_model_version: cicd-stage-failure-logreg-v1
  warning_level: info
  warning_type: none
  recommended_action: check_scm_latency_and_repository_availability
  build_number: 683
  ci_stage: checkout
  signal_name: scm_latency
  is_failure: False

Record 3
  processing_component: spark-mllib-stage-failure-prediction
  ml_model_name: tap-ci-stage-failure-logistic-regression
  ml_model_version: cicd-stage-failu

## Elasticsearch

L'indexer Python consuma `cicd.otel.scored` e salva i documenti finali nell'indice `cicd-observability-events`. Kibana userà proprio questo indice per la dashboard.

In [9]:
index_name = "cicd-observability-events"
count = get_json(f"http://localhost:9200/{index_name}/_count", "elastic", "admine")
search = get_json(f"http://localhost:9200/{index_name}/_search?size=3&sort=@timestamp:desc", "elastic", "admine")

print("Indexed documents:", count["count"])
for hit in search["hits"]["hits"]:
    doc = hit["_source"]
    print(f"\n{doc.get('@timestamp')} | build #{doc.get('build_number')} | {doc.get('ci_stage')}")
    print(" ", doc.get("dashboard_category"), "-", doc.get("warning_type"))
    if doc.get("warning_title"):
        print(" ", doc["warning_title"])

Indexed documents: 3236

2026-05-22T10:52:12.683Z | build #683 | test
  observed_failure - observed_stage_failure
  Stage failed demo-ci-observability #683 test

2026-05-22T10:52:12.683Z | build #682 | deploy
  observability_event - none

2026-05-22T10:52:12.683Z | build #683 | pipeline
  observed_failure - observed_stage_failure
  Stage failed demo-ci-observability #683 pipeline


## Kibana

Kibana è lo strato finale, previsto per l'interazione utente: la dashboard è stata costruita nell'editor di Kibana usando la data view `cicd-observability-events`.

![KPI e segnali](assets/kibana_01_kpi_and_signals.png)
*KPI della pipeline e segnali principali.*

![Avvisi del modello](assets/kibana_02_model_warnings.png)
*Warning generati dal modello.*
In ordine di lettura, come in un fumetto:
1. **Warnings over time:** quanti warning riceviamo dal modello nel tempo?
2. **Warnings by stage:** quanti warning arrivano da ogni stage della pipeline CI/CD?
3. **Model Warning reason:** Treemap col motivo per cui viene emesso il warning
4. **Stage Outcomes:** verde significa che lo stage è andato bene. Giallo significa che il modello aveva previsto un avviso prima del fallimento, rosso significa che lo stage è effettivamente fallito.

![Feed degli avvisi](assets/kibana_03_warning_feed.png)
*Feed degli avvisi e correzioni consigliate dal modello.*
Sempre in ordine di lettura come prima:
1. **Model Warning Causes:** Tabella per vedere qual è il tipo warning più emesso per stage di pipeline. Da usare in tandem con la waffle map (sotto)
2. **Prediction vs Observed failure by stage:** Tabella che ci fa capire, per ogni fallimento (suddiviso per stage di pipeline), quanti effettivamente il modello ne ha previsti prima che arrivassero a fallire.
3. **Model's recommended fix per warning:** Tabella molto importante, dice che azione intraprendere per ogni warning che è stato emesso dal modello per sistemare il problema.
4. **Recommended action distribution:** Waffle map per vedere a colpo d'occhio come sono distribuite le recommended actions da intraprendere (per capire se ci sono stadi della pipeline che stanno andando peggio)

In [11]:
status = get_json("http://localhost:5601/api/status", "elastic", "admine")["status"]["overall"]
print("Kibana status:", status["level"])
print("Details:", status["summary"])
print("Dashboard: http://localhost:5601/app/dashboards#/view/cicd-stage-failure-warnings")

Kibana status: available
Details: All services and plugins are available
Dashboard: http://localhost:5601/app/dashboards#/view/cicd-stage-failure-warnings
